In [1]:
# ============================================================
# 1- Database design in phpMyAdmin -> SQL codes
# ============================================================

In [2]:
# ============================================================
# 2- Phone Number Normalization
# ============================================================

import re

_PERSIAN_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789"
)

_ARABIC_DIGITS = str.maketrans(
    "٠١٢٣٤٥٦٧٨٩",
    "0123456789"
)


def normalize_iran_phone(value) -> str:
    """
    Return an Iranian mobile number in local 10-digit form.

    Example:
        9121234567

    Accepted formats:
        9121234567
        09121234567
        +989121234567
        00989121234567

    Persian and Arabic-Indic digits are also accepted.
    """

    if value is None:
        raise ValueError("Phone number is empty")

    # Normalize Unicode digits first.
    s = str(value).strip()
    s = s.translate(_PERSIAN_DIGITS)
    s = s.translate(_ARABIC_DIGITS)

    # Reject obvious Excel numeric artifacts.
    if re.fullmatch(r"\d+\.0+", s):
        raise ValueError(
            f"Phone number appears to have been converted to a numeric "
            f"Excel value: {value!r}"
        )

    # Keep digits and an optional leading '+'.
    s = re.sub(r"[^\d+]", "", s)

    # Convert international Iranian formats to local format.
    if s.startswith("+98"):
        s = s[3:]
    elif s.startswith("0098"):
        s = s[4:]
    elif s.startswith("98") and len(s) == 12:
        s = s[2:]
    elif s.startswith("0"):
        s = s[1:]

    # Final Iranian mobile validation.
    if not re.fullmatch(r"9\d{9}", s):
        raise ValueError(
            f"Invalid Iranian mobile number: {value!r}"
        )

    return s

In [3]:
# ============================================================
# 3- Import Excel Data into the Database
# ============================================================
from pathlib import Path

import pandas as pd
import mysql.connector

EXCEL_FILE = Path(
    "/home/saeed/repositories/contacts/import_eitaa/contacts-1-44.xlsx"
)


def get_db_connection():
    return mysql.connector.connect(
        host="127.0.0.1",
        port=3306,
        user="reza",
        password="123",
        database="eitaa_import",
        charset="utf8mb4",
    )


def import_excel():
    df = pd.read_excel(
        EXCEL_FILE,
        dtype={
            "Full Name": "string",
            "Phone Number": "string",
        },
    )

    expected = {"Full Name", "Phone Number"}

    if set(df.columns) != expected:
        raise ValueError(
            f"Expected columns {expected}, got {set(df.columns)}"
        )

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        sql = """
            INSERT INTO contacts (
                full_name,
                phone_original,
                phone_normalized,
                status
            )
            VALUES (%s, %s, %s, 'pending')
            ON DUPLICATE KEY UPDATE
                full_name = VALUES(full_name),
                phone_original = VALUES(phone_original)
        """

        processed = 0
        invalid_phones = 0
        empty_names = 0

        for _, row in df.iterrows():

            # ------------------------------------------------
            # Full name
            # ------------------------------------------------
            if pd.isna(row["Full Name"]):
                empty_names += 1
                continue

            full_name = str(row["Full Name"]).strip()

            if not full_name:
                empty_names += 1
                continue

            # ------------------------------------------------
            # Phone number
            # ------------------------------------------------
            if pd.isna(row["Phone Number"]):
                invalid_phones += 1
                print("Invalid phone number: empty Excel cell")
                continue

            phone_original = str(row["Phone Number"]).strip()

            try:
                phone_normalized = normalize_iran_phone(
                    phone_original
                )
            except ValueError as exc:
                invalid_phones += 1
                print(
                    f"Invalid phone number "
                    f"{phone_original!r}: {exc}"
                )
                continue

            # ------------------------------------------------
            # Insert/update database
            # ------------------------------------------------
            cursor.execute(
                sql,
                (
                    full_name,
                    phone_original,
                    phone_normalized,
                ),
            )

            processed += 1

        conn.commit()

        # ----------------------------------------------------
        # Verify database state after import
        # ----------------------------------------------------
        cursor.execute(
            """
            SELECT
                COUNT(*) AS total,
                SUM(status = 'pending') AS pending,
                SUM(status = 'processing') AS processing,
                SUM(status = 'success') AS success,
                SUM(status = 'skipped_existing') AS skipped_existing,
                SUM(status = 'failed') AS failed
            FROM contacts
            """
        )

        stats = cursor.fetchone()

        print()
        print("========== IMPORT RESULT ==========")
        print(f"Excel rows processed : {processed}")
        print(f"Invalid phones       : {invalid_phones}")
        print(f"Empty names          : {empty_names}")
        print()
        print(f"DB total             : {stats[0]}")
        print(f"DB pending           : {stats[1]}")
        print(f"DB processing        : {stats[2]}")
        print(f"DB success           : {stats[3]}")
        print(f"DB skipped_existing  : {stats[4]}")
        print(f"DB failed            : {stats[5]}")
        print("====================================")

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


if __name__ == "__main__":
    import_excel()



========== IMPORT RESULT ==========
Excel rows processed : 44
Invalid phones       : 0
Empty names          : 0

DB total             : 44
DB pending           : 0
DB processing        : 0
DB success           : 36
DB skipped_existing  : 0
DB failed            : 2


In [5]:
# ============================================================
# 4- Selenium Chrome session persistence
# ============================================================
# Clean, isolated Selenium Chrome session

from pathlib import Path
from tempfile import mkdtemp
from typing import Optional
import shutil

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.webdriver import WebDriver

# ChromeDriver

CHROMEDRIVER = (
        Path.cwd()
        / "drivers"
        / "chromedriver"
        / "chromedriver"
)

# Create a completely new temporary Chrome User Data Directory

TEMP_CHROME_PROFILE = Path(
    mkdtemp(prefix="selenium-chrome-")
)

# Close previous Selenium session, if one exists

previous_driver: Optional[WebDriver] = globals().get("driver")

if previous_driver is not None:
    try:
        previous_driver.quit()
        print("\nPrevious Selenium driver closed.")
    except Exception as e:
        print("\nPrevious Selenium driver could not be closed:")
        print(repr(e))

# Chrome options

options = Options()

options.add_argument(
    f"--user-data-dir={TEMP_CHROME_PROFILE}"
)

options.add_argument("--start-maximized")

#  Explicit manual ChromeDriver
#  Selenium Manager is not used for driver discovery.


service = Service(
    executable_path=str(CHROMEDRIVER)
)

# Create new isolated Chrome session


driver: WebDriver = webdriver.Chrome(
    service=service,
    options=options,
)

driver.set_page_load_timeout(60)


In [5]:
# ============================================================
# 5- Close Selenium and remove temporary Chrome profile
# ============================================================

if "driver" in globals() and driver is not None:
    try:
        driver.quit()
        print("Selenium Chrome session closed.")
    finally:
        driver = None

if "TEMP_CHROME_PROFILE" in globals():
    shutil.rmtree(
        TEMP_CHROME_PROFILE,
        ignore_errors=True
    )

    print(
        "Temporary Selenium Chrome profile removed."
    )

Selenium Chrome session closed.
Temporary Selenium Chrome profile removed.


In [6]:
# ============================================================
# 6- Selenium Selectors
# ============================================================

from selenium.webdriver.common.by import By


class Selectors:
    # 1. Hamburger menu
    HAMBURGER_MENU = (
        By.CSS_SELECTOR,
        ".btn-icon.btn-menu-toggle.rp.sidebar-tools-button.is-visible"
    )

    # 2. Contacts menu item
    CONTACTS_BUTTON = (
        By.XPATH,
        "//div[contains(@class, 'btn-menu-item') and .//span[normalize-space()='Contacts']]"
    )

    # 3. "+" button inside Contacts
    ADD_CONTACT_BUTTON = (
        By.XPATH,
        "//button[contains(@class, 'btn-circle') "
        "and contains(@class, 'btn-corner') "
        "and .//span[@data-icon='add']]"
    )

    # 4. First name field
    FIRST_NAME = (
        By.CSS_SELECTOR,
        'div.input-field:first-child > div.input-field-input[contenteditable="true"]'
    )

    # 5. Phone number field
    PHONE = (
        By.CSS_SELECTOR,
        'div.input-field-phone > '
        'div.input-field-input[contenteditable="true"]'
    )

    # 6. Add/Save button
    SAVE_BUTTON = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-primary.btn-color-primary"
    )

    # 7. Toast shown when the phone number is not registered on Eitaa
    UNREGISTERED_TOAST = (
        By.CSS_SELECTOR,
        "div.toast.is-visible"
    )

    # 8. Close Add Contact window ( × button )
    CLOSE_CONTACT_WINDOW = (
        By.CSS_SELECTOR,
        ".popup-create-contact button.btn-icon.popup-close"
    )

In [ ]:
DELETE
ME!
# ============================================================
# TEST - Section 6: Selenium Selectors
# ============================================================

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

WAIT_TIMEOUT = 10

# Test number that produces the Eitaa "not registered" toast
TEST_UNREGISTERED_PHONE = "9111111111"

wait = WebDriverWait(driver, WAIT_TIMEOUT)


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def verify_visible(name, locator):
    """
    Verify that a selector finds a visible element.
    """
    try:
        element = wait.until(
            EC.visibility_of_element_located(locator)
        )

        print(f"✓ {name}")
        return element

    except TimeoutException:
        raise AssertionError(
            f"✗ {name} FAILED\n"
            f"  Locator: {locator}"
        )


def click_element(name, locator):
    """
    Wait for an element to be clickable and click it.
    """
    try:
        element = wait.until(
            EC.element_to_be_clickable(locator)
        )

        element.click()
        print(f"✓ {name} - clicked")
        return element

    except TimeoutException:
        raise AssertionError(
            f"✗ {name} FAILED - element was not clickable\n"
            f"  Locator: {locator}"
        )


# ============================================================
# 1. Verify HAMBURGER_MENU
# ============================================================

print("=" * 70)
print("SECTION 6 - SELENIUM SELECTOR TEST")
print("=" * 70)

print("\n[1] HAMBURGER_MENU")
print("-" * 70)

verify_visible(
    "HAMBURGER_MENU",
    Selectors.HAMBURGER_MENU
)

# ============================================================
# 2. Open hamburger menu
# ============================================================

print("\n[2] Open Hamburger Menu")
print("-" * 70)

click_element(
    "HAMBURGER_MENU",
    Selectors.HAMBURGER_MENU
)

# ============================================================
# 3. Verify CONTACTS_BUTTON
# ============================================================

print("\n[3] CONTACTS_BUTTON")
print("-" * 70)

verify_visible(
    "CONTACTS_BUTTON",
    Selectors.CONTACTS_BUTTON
)

# ============================================================
# 4. Open Contacts
# ============================================================

print("\n[4] Open Contacts")
print("-" * 70)

click_element(
    "CONTACTS_BUTTON",
    Selectors.CONTACTS_BUTTON
)

# ============================================================
# 5. Verify ADD_CONTACT_BUTTON
# ============================================================

print("\n[5] ADD_CONTACT_BUTTON")
print("-" * 70)

verify_visible(
    "ADD_CONTACT_BUTTON",
    Selectors.ADD_CONTACT_BUTTON
)

# ============================================================
# 6. Open Add Contact dialog
# ============================================================

print("\n[6] Open Add Contact Dialog")
print("-" * 70)

click_element(
    "ADD_CONTACT_BUTTON",
    Selectors.ADD_CONTACT_BUTTON
)

# ============================================================
# 7. Verify FIRST_NAME
# ============================================================

print("\n[7] FIRST_NAME")
print("-" * 70)

first_name = verify_visible(
    "FIRST_NAME",
    Selectors.FIRST_NAME
)

# ============================================================
# 8. Verify PHONE
# ============================================================

print("\n[8] PHONE")
print("-" * 70)

phone = verify_visible(
    "PHONE",
    Selectors.PHONE
)

# ============================================================
# 9. Verify SAVE_BUTTON
# ============================================================

print("\n[9] SAVE_BUTTON")
print("-" * 70)

verify_visible(
    "SAVE_BUTTON",
    Selectors.SAVE_BUTTON
)

# ============================================================
# 10. Verify CLOSE_CONTACT_WINDOW
# ============================================================

print("\n[10] CLOSE_CONTACT_WINDOW")
print("-" * 70)

verify_visible(
    "CLOSE_CONTACT_WINDOW",
    Selectors.CLOSE_CONTACT_WINDOW
)

# ============================================================
# 11. Enter test phone number
# ============================================================

print("\n[11] Enter Test Phone Number")
print("-" * 70)

phone.click()
phone.send_keys(TEST_UNREGISTERED_PHONE)

print(
    f"✓ PHONE accepted test number: "
    f"{TEST_UNREGISTERED_PHONE}"
)

# ============================================================
# 12. Enter test first name
# ============================================================

print("\n[12] Enter Test First Name")
print("-" * 70)

first_name.click()
first_name.send_keys("SelectorTest")

print("✓ FIRST_NAME accepted test text")

# ============================================================
# 13. Click SAVE
# ============================================================

print("\n[13] SAVE_BUTTON")
print("-" * 70)

click_element(
    "SAVE_BUTTON",
    Selectors.SAVE_BUTTON
)

# ============================================================
# 14. Detect transient UNREGISTERED_TOAST
#
# IMPORTANT:
# The toast exists only briefly, so we poll the DOM
# frequently instead of using a long normal WebDriverWait.
# ============================================================

print("\n[14] UNREGISTERED_TOAST")
print("-" * 70)

toast_found = False

toast_locator = Selectors.UNREGISTERED_TOAST

toast_deadline = time.monotonic() + 5

while time.monotonic() < toast_deadline:

    try:
        toast_elements = driver.find_elements(
            *toast_locator
        )

        for toast in toast_elements:

            if toast.is_displayed():

                toast_found = True

                print("✓ UNREGISTERED_TOAST appeared")

                try:
                    print(
                        f"  Toast text: "
                        f"{toast.text!r}"
                    )
                except Exception:
                    pass

                break

        if toast_found:
            break

    except Exception:
        pass

    time.sleep(0.05)

if not toast_found:
    raise AssertionError(
        "✗ UNREGISTERED_TOAST FAILED\n"
        "The toast did not appear within the 5-second "
        "monitoring window."
    )

# ============================================================
# 15. CLOSE TOAST
#
# The same CLOSE_CONTACT_WINDOW selector is used.
# The first click dismisses the toast.
# ============================================================

print("\n[15] Close Toast")
print("-" * 70)

click_element(
    "CLOSE_CONTACT_WINDOW - first click (dismiss toast)",
    Selectors.CLOSE_CONTACT_WINDOW
)

print("✓ Toast dismissed")

# ------------------------------------------------------------
# Give Eitaa a very short moment to update the UI.
# ------------------------------------------------------------

time.sleep(0.2)

# ============================================================
# 16. CLOSE ADD CONTACT WINDOW
#
# The same selector must be clicked a second time.
# ============================================================

print("\n[16] Close Add Contact Window")
print("-" * 70)

click_element(
    "CLOSE_CONTACT_WINDOW - second click (close window)",
    Selectors.CLOSE_CONTACT_WINDOW
)

print("✓ Add Contact window close action executed")

# ============================================================
# 17. Verify Add Contact window is actually closed
# ============================================================

print("\n[17] Verify Add Contact Window Closed")
print("-" * 70)

try:
    WebDriverWait(driver, 5).until(
        EC.invisibility_of_element_located(
            Selectors.CLOSE_CONTACT_WINDOW
        )
    )

    print("✓ Add Contact window is closed")

except TimeoutException:
    raise AssertionError(
        "✗ Add Contact window is still visible after "
        "the second CLOSE_CONTACT_WINDOW click."
    )

# ============================================================
# 18. Final result
# ============================================================

print("\n" + "=" * 70)
print("SECTION 6 TEST PASSED")
print("=" * 70)

print("""
All Section 6 Selenium selectors were successfully verified:

✓ HAMBURGER_MENU
✓ CONTACTS_BUTTON
✓ ADD_CONTACT_BUTTON
✓ FIRST_NAME
✓ PHONE
✓ SAVE_BUTTON
✓ UNREGISTERED_TOAST
✓ CLOSE_CONTACT_WINDOW

The transient UNREGISTERED_TOAST was detected successfully.

Cleanup completed:

✓ First CLOSE_CONTACT_WINDOW click dismissed the toast
✓ Second CLOSE_CONTACT_WINDOW click closed the Add Contact window
✓ Add Contact window disappearance was verified
""")


In [7]:
# ============================================================
# 7- Reliable waits
# ============================================================

# Use explicit waits instead of time.sleep().
#
# The purpose of this section is to provide reusable,
# reliable synchronization helpers for the Selenium code
# used in later sections.
#
# These helpers are designed for situations such as:
#   - waiting for Chrome/Eitaa elements to appear
#   - waiting for an element to become visible
#   - waiting for an element to become clickable
#   - waiting for an element/popup to disappear
#   - waiting for a specific Selenium condition
#   - handling temporary/stale DOM elements
#
# ============================================================

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    StaleElementReferenceException,
    NoSuchElementException,
    ElementClickInterceptedException,
)

# ------------------------------------------------------------
# General configuration
# ------------------------------------------------------------

WAIT_SECONDS = 30


# ------------------------------------------------------------
# 1. Create a WebDriverWait object
# ------------------------------------------------------------

def get_wait(driver, timeout=WAIT_SECONDS):
    """
    Return a WebDriverWait instance.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    WebDriverWait
    """
    return WebDriverWait(
        driver,
        timeout,
        poll_frequency=0.2,
        ignored_exceptions=(
            NoSuchElementException,
            StaleElementReferenceException,
        ),
    )


# ------------------------------------------------------------
# 2. Wait until an element exists in the DOM
# ------------------------------------------------------------

def wait_present(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present in the DOM.

    The element does not necessarily have to be visible.

    Useful when:
        - the element exists but is not yet displayed
        - another wait will handle visibility/clickability
    """
    return get_wait(driver, timeout).until(
        EC.presence_of_element_located(locator)
    )


# ------------------------------------------------------------
# 3. Wait until an element is visible
# ------------------------------------------------------------

def wait_visible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is present and visible.

    Returns
    -------
    WebElement
    """
    return get_wait(driver, timeout).until(
        EC.visibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 4. Wait until an element is clickable
# ------------------------------------------------------------

def wait_clickable(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is visible and enabled/clickable.

    This should normally be used before clicking buttons,
    links, menu items, etc.
    """
    return get_wait(driver, timeout).until(
        EC.element_to_be_clickable(locator)
    )


# ------------------------------------------------------------
# 5. Wait until an element disappears
# ------------------------------------------------------------

def wait_invisible(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is no longer visible.

    This is particularly useful for:
        - closing popups
        - waiting for dialogs to disappear
        - waiting for loading overlays to disappear

    Note:
        Selenium's invisibility_of_element_located() also
        succeeds if the element is removed from the DOM.
    """
    return get_wait(driver, timeout).until(
        EC.invisibility_of_element_located(locator)
    )


# ------------------------------------------------------------
# 6. Wait until an element disappears from the DOM
# ------------------------------------------------------------

def wait_stale(driver, element, timeout=WAIT_SECONDS):
    """
    Wait until a previously obtained WebElement becomes stale.

    Useful when an existing element is replaced/removed after
    an operation such as saving a form.
    """
    return get_wait(driver, timeout).until(
        EC.staleness_of(element)
    )


# ------------------------------------------------------------
# 7. Wait for a locator to contain specific text
# ------------------------------------------------------------

def wait_text(driver, locator, text, timeout=WAIT_SECONDS):
    """
    Wait until an element contains the specified text.
    """
    return get_wait(driver, timeout).until(
        EC.text_to_be_present_in_element(
            locator,
            text,
        )
    )


# ------------------------------------------------------------
# 8. Wait for a specific attribute value
# ------------------------------------------------------------

def wait_attribute(
        driver,
        locator,
        attribute,
        value,
        timeout=WAIT_SECONDS,
):
    """
    Wait until an element's attribute contains the specified value.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    locator : tuple
        Selenium locator.

    attribute : str
        Name of the HTML attribute to inspect.

    value : str
        Value that must be present in the attribute.

    timeout : int | float
        Maximum number of seconds to wait.

    Returns
    -------
    bool
        True when the attribute contains the specified value.
    """

    def attribute_contains(driver):
        try:
            element = driver.find_element(*locator)
            attribute_value = element.get_attribute(attribute)

            if attribute_value is None:
                return False

            return value in attribute_value

        except (
                NoSuchElementException,
                StaleElementReferenceException,
        ):
            return False

    return get_wait(driver, timeout).until(
        attribute_contains
    )


# ------------------------------------------------------------
# 9. Wait until an element is selected
# ------------------------------------------------------------

def wait_selected(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until a selectable element is selected.
    """
    return get_wait(driver, timeout).until(
        EC.element_located_to_be_selected(locator)
    )


# ------------------------------------------------------------
# 10. Wait for a custom condition
# ------------------------------------------------------------

def wait_until(
        driver,
        condition,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a custom Selenium condition returns a truthy
    value.

    The condition receives the driver.

    Example:

        wait_until(
            driver,
            lambda d: some_condition(d)
        )

    This is useful when Selenium's built-in expected
    conditions are not sufficient.
    """
    return get_wait(driver, timeout).until(condition)


# ------------------------------------------------------------
# 11. Safe click helper
# ------------------------------------------------------------

def wait_and_click(driver, locator, timeout=WAIT_SECONDS):
    """
    Wait until an element is clickable and then click it.

    Returns
    -------
    WebElement
        The element that was clicked.
    """
    element = wait_clickable(
        driver,
        locator,
        timeout,
    )

    element.click()

    return element


# ------------------------------------------------------------
# 12. Wait for a popup to close
# ------------------------------------------------------------

def wait_popup_closed(
        driver,
        popup_locator,
        timeout=WAIT_SECONDS,
):
    """
    Wait until a popup/dialog is no longer visible.

    This is a convenience wrapper around
    wait_invisible().

    It is intended for situations such as the Eitaa
    Add Contact popup.
    """
    return wait_invisible(
        driver,
        popup_locator,
        timeout,
    )


# ------------------------------------------------------------
# 13. Wait for an element and then enter text
# ------------------------------------------------------------

def wait_and_type(
        driver,
        locator,
        text,
        timeout=WAIT_SECONDS,
        clear_first=True,
):
    """
    Wait for an input field to become visible and type text.

    Parameters
    ----------
    driver : WebDriver
        Active Selenium WebDriver.

    locator : tuple
        Selenium locator, for example:
            (By.CSS_SELECTOR, "...")

    text : str
        Text to enter.

    timeout : int | float
        Maximum number of seconds to wait.

    clear_first : bool
        If True, clear the input before typing.

    Returns
    -------
    WebElement
        The input element.
    """
    element = wait_visible(
        driver,
        locator,
        timeout,
    )

    if clear_first:
        element.clear()

    element.send_keys(text)

    return element

In [ ]:
DELETE
ME!
# ============================================================
# TEST — Section 7: Reliable waits
# ============================================================
#
# This test verifies all reusable wait helpers defined in
# Section 7.
#
# The test uses a temporary local HTML page, so it does NOT:
#   - modify Eitaa
#   - add any contacts
#   - modify the MySQL database
#   - require an Eitaa login
#
# It uses the existing Selenium driver from Section 4.
#
# ============================================================

from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException

print("=" * 60)
print("SECTION 7 TEST — RELIABLE WAITS")
print("=" * 60)

# ------------------------------------------------------------
# 1. Verify required Section 7 objects exist
# ------------------------------------------------------------

required_functions = [
    "get_wait",
    "wait_present",
    "wait_visible",
    "wait_clickable",
    "wait_invisible",
    "wait_stale",
    "wait_text",
    "wait_attribute",
    "wait_selected",
    "wait_until",
    "wait_and_click",
    "wait_popup_closed",
    "wait_and_type",
]

missing_functions = [
    name
    for name in required_functions
    if name not in globals() or not callable(globals()[name])
]

if missing_functions:
    raise AssertionError(
        "Missing Section 7 functions: "
        + ", ".join(missing_functions)
    )

print("✓ All Section 7 functions are available")

# ------------------------------------------------------------
# 2. Verify Selenium driver exists
# ------------------------------------------------------------

if "driver" not in globals() or driver is None:
    raise RuntimeError(
        "Selenium driver was not found. "
        "Run Section 4 before running this test."
    )

print("✓ Selenium driver is available")

# ------------------------------------------------------------
# 3. Create a temporary test page
# ------------------------------------------------------------

test_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Section 7 Wait Test</title>
</head>

<body>

    <div id="present-element">
        Present element
    </div>

    <div id="visible-element">
        Visible element
    </div>

    <button id="click-button" type="button">
        Click me
    </button>

    <div id="text-element">
        Waiting for test text
    </div>

    <div id="attribute-element" class="test-element">
        Attribute test
    </div>

    <input
        id="text-input"
        type="text"
        value="old value"
    >

    <select id="select-element">
        <option value="one">One</option>
        <option value="two" selected>Two</option>
    </select>

    <div id="popup">
        Test popup
    </div>

    <div id="stale-element">
        This element will become stale
    </div>

    <div id="custom-element">
        Custom condition
    </div>

    <script>

        // Make the text condition available shortly after load.
        setTimeout(function () {
            document.getElementById("text-element").textContent =
                "Expected test text";
        }, 300);

        // Change the class attribute shortly after load.
        setTimeout(function () {
            document.getElementById("attribute-element")
                .className = "test-element active";
        }, 300);

        // Hide the popup shortly after load.
        setTimeout(function () {
            document.getElementById("popup").style.display = "none";
        }, 300);

        // Make the custom condition true shortly after load.
        setTimeout(function () {
            document.getElementById("custom-element")
                .setAttribute("data-ready", "true");
        }, 300);

    </script>

</body>
</html>
"""

# Load the HTML directly in Chrome.
driver.get(
    "data:text/html;charset=utf-8,"
    + test_html
)

# ------------------------------------------------------------
# 4. Define test locators
# ------------------------------------------------------------

PRESENT = (By.ID, "present-element")
VISIBLE = (By.ID, "visible-element")
CLICK_BUTTON = (By.ID, "click-button")
TEXT_ELEMENT = (By.ID, "text-element")
ATTRIBUTE_ELEMENT = (By.ID, "attribute-element")
TEXT_INPUT = (By.ID, "text-input")
SELECT_ELEMENT = (By.ID, "select-element")
POPUP = (By.ID, "popup")
STALE_ELEMENT = (By.ID, "stale-element")
CUSTOM_ELEMENT = (By.ID, "custom-element")

# ------------------------------------------------------------
# 5. Test get_wait()
# ------------------------------------------------------------

wait_object = get_wait(driver, timeout=5)

assert isinstance(
    wait_object,
    type(get_wait(driver, timeout=1))
)

print("✓ get_wait()")

# ------------------------------------------------------------
# 6. Test wait_present()
# ------------------------------------------------------------

element = wait_present(
    driver,
    PRESENT,
    timeout=5,
)

assert element is not None
assert element.get_attribute("id") == "present-element"

print("✓ wait_present()")

# ------------------------------------------------------------
# 7. Test wait_visible()
# ------------------------------------------------------------

element = wait_visible(
    driver,
    VISIBLE,
    timeout=5,
)

assert element.is_displayed()

print("✓ wait_visible()")

# ------------------------------------------------------------
# 8. Test wait_clickable()
# ------------------------------------------------------------

element = wait_clickable(
    driver,
    CLICK_BUTTON,
    timeout=5,
)

assert element.is_displayed()
assert element.is_enabled()

print("✓ wait_clickable()")

# ------------------------------------------------------------
# 9. Test wait_and_click()
# ------------------------------------------------------------

clicked_element = wait_and_click(
    driver,
    CLICK_BUTTON,
    timeout=5,
)

assert clicked_element.get_attribute("id") == "click-button"

print("✓ wait_and_click()")

# ------------------------------------------------------------
# 10. Test wait_text()
# ------------------------------------------------------------

wait_text(
    driver,
    TEXT_ELEMENT,
    "Expected test text",
    timeout=5,
)

actual_text = driver.find_element(
    *TEXT_ELEMENT
).text

assert "Expected test text" in actual_text

print("✓ wait_text()")

# ------------------------------------------------------------
# 11. Test wait_attribute()
# ------------------------------------------------------------

wait_attribute(
    driver,
    ATTRIBUTE_ELEMENT,
    "class",
    "active",
    timeout=5,
)

attribute_value = driver.find_element(
    *ATTRIBUTE_ELEMENT
).get_attribute("class")

assert "active" in attribute_value

print("✓ wait_attribute()")

# ------------------------------------------------------------
# 12. Test wait_selected()
# ------------------------------------------------------------

selected = wait_selected(
    driver,
    (By.CSS_SELECTOR, "#select-element option:checked"),
    timeout=5,
)

assert selected is not False

print("✓ wait_selected()")

# ------------------------------------------------------------
# 13. Test wait_and_type()
# ------------------------------------------------------------

typed_element = wait_and_type(
    driver,
    TEXT_INPUT,
    "Section 7 test",
    timeout=5,
)

assert typed_element.get_attribute("value") == "Section 7 test"

print("✓ wait_and_type()")

# ------------------------------------------------------------
# 14. Test wait_invisible()
# ------------------------------------------------------------

wait_invisible(
    driver,
    POPUP,
    timeout=5,
)

popup = driver.find_element(*POPUP)

assert not popup.is_displayed()

print("✓ wait_invisible()")

# ------------------------------------------------------------
# 15. Test wait_popup_closed()
# ------------------------------------------------------------

# The popup is already invisible, so this verifies that
# wait_popup_closed() correctly delegates to wait_invisible().

wait_popup_closed(
    driver,
    POPUP,
    timeout=5,
)

print("✓ wait_popup_closed()")

# ------------------------------------------------------------
# 16. Test wait_stale()
# ------------------------------------------------------------

# Create a fresh element specifically for this test.
driver.execute_script("""
    const element = document.createElement("div");
    element.id = "stale-test-element";
    element.textContent = "This element will become stale";
    document.body.appendChild(element);
""")

STALE_TEST_ELEMENT = (
    By.ID,
    "stale-test-element",
)

# Obtain the WebElement before it is removed.
stale_element = wait_present(
    driver,
    STALE_TEST_ELEMENT,
    timeout=5,
)

# Remove the element after a short delay.
driver.execute_script("""
    const element =
        document.getElementById("stale-test-element");

    setTimeout(function () {
        if (element) {
            element.remove();
        }
    }, 300);
""")

# Wait until the previously obtained WebElement becomes stale.
wait_stale(
    driver,
    stale_element,
    timeout=5,
)

print("✓ wait_stale()")

# ------------------------------------------------------------
# 17. Test wait_until()
# ------------------------------------------------------------

wait_until(
    driver,
    lambda d: d.find_element(
        *CUSTOM_ELEMENT
    ).get_attribute("data-ready") == "true",
    timeout=5,
)

assert driver.find_element(
    *CUSTOM_ELEMENT
).get_attribute("data-ready") == "true"

print("✓ wait_until()")

# ------------------------------------------------------------
# 18. Final result
# ------------------------------------------------------------

print()
print("=" * 60)
print("SECTION 7 TEST PASSED")
print("=" * 60)
print()
print("All Section 7 reliable-wait helpers passed successfully.")
print("The Selenium driver is still available for later sections.")

In [8]:
# ============================================================
# 8- Add Contact to Eitaa
# ============================================================

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
)

# ------------------------------------------------------------
# Eitaa result
# ------------------------------------------------------------

UNREGISTERED_MESSAGE = (
    "The person with this phone number is not registered on Eitaa yet."
)


# ------------------------------------------------------------
# 8-1 Database result update
# ------------------------------------------------------------

def _update_contact_result(
        contact_id,
        status,
        error_message=None,
):
    """
    Update the final result of the current single attempt.

    Section 8 does NOT modify the attempts counter.
    Retry/attempt management belongs exclusively to Section 10.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE contacts
            SET
                status = %s,
                error_message = %s,
                completed_at = NOW()
            WHERE id = %s
            """,
            (
                status,
                error_message,
                contact_id,
            ),
        )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# 8-2 Contacts page detection
# ------------------------------------------------------------

def _contacts_page_is_open(driver):
    """
    Return True when the Add Contact button is already visible.

    This prevents Section 8 from blindly clicking the hamburger
    menu when the Contacts area is already open.
    """

    try:
        elements = driver.find_elements(
            *Selectors.ADD_CONTACT_BUTTON
        )

        return any(
            element.is_displayed()
            for element in elements
        )

    except WebDriverException:
        return False


# ------------------------------------------------------------
# 8-3 Open Contacts area when necessary
# ------------------------------------------------------------

def _open_contacts_area(driver):
    """
    Open the Contacts area if it is not already open.
    """

    if _contacts_page_is_open(driver):
        return

    wait_and_click(
        driver,
        Selectors.HAMBURGER_MENU,
    )

    wait_and_click(
        driver,
        Selectors.CONTACTS_BUTTON,
    )

    wait_visible(
        driver,
        Selectors.ADD_CONTACT_BUTTON,
    )


# ------------------------------------------------------------
# 8-4 Detect the unregistered-phone result
# ------------------------------------------------------------

def _get_unregistered_result(driver):
    """
    Return True when the Eitaa unregistered-phone toast is visible
    and contains the expected message.

    Return False when the toast is not currently present.
    """

    try:
        toast_elements = driver.find_elements(
            *Selectors.UNREGISTERED_TOAST
        )

        for toast in toast_elements:
            if not toast.is_displayed():
                continue

            text = toast.text.strip()

            if UNREGISTERED_MESSAGE in text:
                return True

    except WebDriverException:
        return False

    return False


# ------------------------------------------------------------
# 8-5 Detect result after Save
# ------------------------------------------------------------

def _wait_for_add_contact_result(driver):
    """
    Wait for exactly one result of the current Add Contact attempt.

    Possible return values:
        "skipped"  -> Eitaa reported that the phone is unregistered
        "success"  -> Add Contact window closed without that message

    If neither state can be determined within the wait period,
    wait_until() raises TimeoutException and the caller treats
    the attempt as failed.
    """

    def result_condition(d):
        # ----------------------------------------------------
        # First priority: unregistered phone
        #
        # This must be checked before treating popup closure as
        # success because the toast is the definitive skipped
        # result for an unregistered number.
        # ----------------------------------------------------
        if _get_unregistered_result(d):
            return "skipped"

        # ----------------------------------------------------
        # Registered/success case:
        #
        # After a successful Add Contact operation, the
        # Add Contact window is expected to close.
        #
        # CLOSE_CONTACT_WINDOW is the existing Section 6
        # selector and therefore is used as the popup-state
        # indicator without inventing a new selector.
        # ----------------------------------------------------
        try:
            close_buttons = d.find_elements(
                *Selectors.CLOSE_CONTACT_WINDOW
            )

            visible_close_buttons = [
                button
                for button in close_buttons
                if button.is_displayed()
            ]

            if not visible_close_buttons:
                return "success"

        except WebDriverException:
            return False

        return False

    return wait_until(
        driver,
        result_condition,
    )


# ------------------------------------------------------------
# 8-6 Close Add Contact window
# ------------------------------------------------------------

def _close_add_contact_window(driver):
    """
    Close the Add Contact window using the existing × button.

    The function does not blindly click twice.

    It first waits for the existing close button and clicks it.
    If the popup is still visible afterwards, it conditionally
    performs one additional click.
    """

    # --------------------------------------------------------
    # First close attempt
    # --------------------------------------------------------

    wait_and_click(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    # --------------------------------------------------------
    # Wait for the popup to close.
    #
    # If it closes successfully, no second click is performed.
    # --------------------------------------------------------

    try:
        wait_popup_closed(
            driver,
            Selectors.CLOSE_CONTACT_WINDOW,
            timeout=5,
        )
        return

    except TimeoutException:
        pass

    # --------------------------------------------------------
    # The popup is still visible.
    #
    # Only now perform the second click required by the
    # Eitaa UI behavior described for the unregistered case.
    # --------------------------------------------------------

    wait_and_click(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    wait_popup_closed(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )


# ------------------------------------------------------------
# 8-7 Best-effort cleanup after an operational failure
# ------------------------------------------------------------

def _close_add_contact_window_if_open(driver):
    """
    Best-effort cleanup after a failed attempt.

    Cleanup errors are intentionally ignored because the original
    Selenium/UI error must remain the result of this attempt.
    """

    try:
        elements = driver.find_elements(
            *Selectors.CLOSE_CONTACT_WINDOW
        )

        if not any(
                element.is_displayed()
                for element in elements
        ):
            return

        wait_and_click(
            driver,
            Selectors.CLOSE_CONTACT_WINDOW,
            timeout=5,
        )

        try:
            wait_popup_closed(
                driver,
                Selectors.CLOSE_CONTACT_WINDOW,
                timeout=5,
            )
        except TimeoutException:
            pass

    except Exception:
        pass


# ------------------------------------------------------------
# 8-8 Enter phone number with Eitaa country prefix
# ------------------------------------------------------------

from selenium.webdriver.common.keys import Keys


def _enter_phone_number(driver, phone_normalized):
    """
    Clear the Eitaa phone field, verify that the previous phone
    number/country prefix has been removed, then enter:

        98 + normalized phone number

    The '+' sign is NOT entered manually because Eitaa adds it
    automatically.
    """

    # --------------------------------------------------------
    # 1. Wait for the phone field
    # --------------------------------------------------------

    phone_field = wait_visible(
        driver,
        Selectors.PHONE,
    )

    # --------------------------------------------------------
    # 2. Clear the existing contents
    #
    # Use both Selenium's clear() and keyboard selection.
    # This is more reliable for phone inputs that use client-side
    # formatting/masking.
    # --------------------------------------------------------

    phone_field.click()

    phone_field.clear()

    phone_field.send_keys(
        Keys.CONTROL,
        "a",
    )

    phone_field.send_keys(
        Keys.BACKSPACE,
    )

    # --------------------------------------------------------
    # 3. Verify that the old 98 prefix is no longer present
    #
    # We intentionally do NOT require value == "" because Eitaa's
    # phone field may maintain its own country-code state.
    # The important condition is that the old phone digits and
    # country prefix are no longer present in the editable value.
    # --------------------------------------------------------

    def phone_field_is_cleared(d):
        try:
            element = d.find_element(
                *Selectors.PHONE
            )

            value = element.get_attribute("value")

            if value is None:
                return True

            value = value.strip()

            # The field must not contain the old country prefix
            # as an entered phone-number value.
            #
            # Eitaa may keep its own "+" / country-code UI state,
            # so an absolutely empty value is not required here.

            return (
                    value == ""
                    or value == "+"
            )

        except WebDriverException:
            return False

    wait_until(
        driver,
        phone_field_is_cleared,
        timeout=5,
    )

    # --------------------------------------------------------
    # 4. Prepare normalized phone number
    # --------------------------------------------------------

    phone_number = str(phone_normalized).strip()

    if phone_number.startswith("98"):
        raise ValueError(
            "phone_normalized must not contain the country code 98."
        )

    # --------------------------------------------------------
    # 5. Enter 98 + normalized phone number
    #
    # Do NOT enter '+' manually.
    # Eitaa adds '+' automatically.
    # --------------------------------------------------------

    phone_field = wait_visible(
        driver,
        Selectors.PHONE,
    )

    phone_field.send_keys(
        "98" + phone_number
    )


# ============================================================
# 8-9 Main Section 8 function
# ============================================================

def add_contact(
        contact_id,
        full_name,
        phone_normalized,
):
    """
    Process exactly ONE contact in exactly ONE attempt.

    Parameters
    ----------
    contact_id : int
        Primary key of the contact in the contacts table.

    full_name : str
        Contact name.

    phone_normalized : str
        Normalized Iranian phone number already stored in the
        contacts table.

    Returns
    -------
    str
        One of:

            "success"
            "skipped"
            "failed"

    Important
    ---------
    This function contains NO retry mechanism.

    A failed result is returned to Section 10, which is
    responsible for deciding whether another attempt is needed.
    """

    try:
        # ----------------------------------------------------
        # 1. Open Contacts area if necessary
        # ----------------------------------------------------

        _open_contacts_area(driver)

        # ----------------------------------------------------
        # 2. Open Add Contact window
        # ----------------------------------------------------

        wait_and_click(
            driver,
            Selectors.ADD_CONTACT_BUTTON,
        )

        # ----------------------------------------------------
        # 3. Enter contact name
        # ----------------------------------------------------

        wait_and_type(
            driver,
            Selectors.FIRST_NAME,
            str(full_name),
        )

        # ----------------------------------------------------
        # 4. Enter normalized phone number
        #
        # IMPORTANT:
        # Do NOT use wait_and_type() here because it clears the
        # field and immediately types the normalized number.
        #
        # Instead:
        #   1. clear the field
        #   2. verify it is empty
        #   3. enter 98
        #   4. enter the normalized number
        #
        # The '+' sign is added automatically by Eitaa.
        # ----------------------------------------------------

        _enter_phone_number(
            driver,
            phone_normalized,
        )

        # ----------------------------------------------------
        # 5. Submit Add Contact form
        # ----------------------------------------------------

        wait_and_click(
            driver,
            Selectors.SAVE_BUTTON,
        )

        # ----------------------------------------------------
        # 6. Determine the result of THIS attempt
        #
        # No retry occurs here.
        # ----------------------------------------------------

        result = _wait_for_add_contact_result(driver)

        # ----------------------------------------------------
        # 7. Registered contact
        # ----------------------------------------------------

        if result == "success":
            _update_contact_result(
                contact_id=contact_id,
                status="success",
                error_message=None,
            )

            return "success"

        # ----------------------------------------------------
        # 8. Unregistered phone number
        # ----------------------------------------------------

        if result == "skipped":
            # The unregistered toast has already been reliably
            # detected. Close the Add Contact operation using
            # the existing close-button mechanism.
            _close_add_contact_window(driver)

            _update_contact_result(
                contact_id=contact_id,
                status="skipped",
                error_message=None,
            )

            return "skipped"

        # ----------------------------------------------------
        # 9.Defensive guard
        # ----------------------------------------------------

        raise RuntimeError(
            f"Unexpected Section 8 result: {result!r}"
        )

    except Exception as exc:

        # ----------------------------------------------------
        # Any Selenium/UI/operational/database error belongs
        # to the FAILED state.
        #
        # Section 8 does NOT retry.
        # Section 10 owns retry decisions.
        # ----------------------------------------------------

        error_message = (
            f"{type(exc).__name__}: {exc}"
        )

        # Best-effort UI cleanup.
        _close_add_contact_window_if_open(driver)

        # Record the failed result.
        #
        # If the database itself is unavailable, the database
        # update can also fail. The original exception remains
        # the failure represented by this attempt.
        try:
            _update_contact_result(
                contact_id=contact_id,
                status="failed",
                error_message=error_message,
            )
        except Exception as db_exc:
            print(
                "Section 8 could not update the failed database state:"
            )
            print(
                f"{type(db_exc).__name__}: {db_exc}"
            )

        print(
            f"Section 8 failed for contact_id={contact_id}: "
            f"{error_message}"
        )

        return "failed"

In [ ]:
DELETE
ME!
# ------------------------------------------------------------
# TEST — Section 8-1 Database result update
# ------------------------------------------------------------

# This test verifies that _update_contact_result() correctly:
#
#   1. Updates status.
#   2. Updates error_message.
#   3. Sets completed_at.
#   4. Does NOT modify attempts.
#
# The test automatically restores the contact to its original
# database state when finished.
#
# IMPORTANT:
# This test does NOT interact with Selenium or Eitaa.
# It tests only the database-update function.


# ------------------------------------------------------------
# 1. Select one existing contact
# ------------------------------------------------------------

conn = get_db_connection()

try:
    cursor = conn.cursor(dictionary=True)

    cursor.execute("""
        SELECT
            id,
            full_name,
            phone_normalized,
            status,
            attempts,
            error_message,
            completed_at
        FROM contacts
        ORDER BY id
        LIMIT 1
    """)

    original = cursor.fetchone()

finally:
    cursor.close()
    conn.close()

if original is None:
    raise RuntimeError(
        "TEST FAILED: No contacts exist in the contacts table."
    )

contact_id = original["id"]

print("Test contact:")
print(f"  id       : {original['id']}")
print(f"  name     : {original['full_name']}")
print(f"  phone    : {original['phone_normalized']}")
print(f"  status   : {original['status']}")
print(f"  attempts : {original['attempts']}")

# ------------------------------------------------------------
# 2. Run the actual test
# ------------------------------------------------------------

try:

    # --------------------------------------------------------
    # Test A — success result
    # --------------------------------------------------------

    _update_contact_result(
        contact_id=contact_id,
        status="success",
        error_message=None,
    )

    # --------------------------------------------------------
    # Read database after success update
    # --------------------------------------------------------

    conn = get_db_connection()

    try:
        cursor = conn.cursor(dictionary=True)

        cursor.execute("""
            SELECT
                status,
                attempts,
                error_message,
                completed_at
            FROM contacts
            WHERE id = %s
        """, (contact_id,))

        result_success = cursor.fetchone()

    finally:
        cursor.close()
        conn.close()

    print("\nAfter success update:")
    print(f"  status        : {result_success['status']}")
    print(f"  attempts      : {result_success['attempts']}")
    print(f"  error_message : {result_success['error_message']}")
    print(f"  completed_at  : {result_success['completed_at']}")

    # --------------------------------------------------------
    # Verify success result
    # --------------------------------------------------------

    assert result_success["status"] == "success", (
        "TEST FAILED: status was not updated to 'success'."
    )

    assert result_success["error_message"] is None, (
        "TEST FAILED: error_message should be None for success."
    )

    assert result_success["completed_at"] is not None, (
        "TEST FAILED: completed_at was not set."
    )

    assert result_success["attempts"] == original["attempts"], (
        "TEST FAILED: attempts was modified by Section 8-1."
    )

    # --------------------------------------------------------
    # Test B — failed result
    # --------------------------------------------------------

    test_error = "TEST ERROR — Section 8-1"

    _update_contact_result(
        contact_id=contact_id,
        status="failed",
        error_message=test_error,
    )

    # --------------------------------------------------------
    # Read database after failed update
    # --------------------------------------------------------

    conn = get_db_connection()

    try:
        cursor = conn.cursor(dictionary=True)

        cursor.execute("""
            SELECT
                status,
                attempts,
                error_message,
                completed_at
            FROM contacts
            WHERE id = %s
        """, (contact_id,))

        result_failed = cursor.fetchone()

    finally:
        cursor.close()
        conn.close()

    print("\nAfter failed update:")
    print(f"  status        : {result_failed['status']}")
    print(f"  attempts      : {result_failed['attempts']}")
    print(f"  error_message : {result_failed['error_message']}")
    print(f"  completed_at  : {result_failed['completed_at']}")

    # --------------------------------------------------------
    # Verify failed result
    # --------------------------------------------------------

    assert result_failed["status"] == "failed", (
        "TEST FAILED: status was not updated to 'failed'."
    )

    assert result_failed["error_message"] == test_error, (
        "TEST FAILED: error_message was not stored correctly."
    )

    assert result_failed["completed_at"] is not None, (
        "TEST FAILED: completed_at was not set for failed result."
    )

    assert result_failed["attempts"] == original["attempts"], (
        "TEST FAILED: attempts was modified by Section 8-1."
    )

    print("\n============================================================")
    print("TEST PASSED — Section 8-1 Database result update")
    print("============================================================")


finally:

    # --------------------------------------------------------
    # 3. ALWAYS restore the original database state
    # --------------------------------------------------------

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute("""
            UPDATE contacts
            SET
                status = %s,
                attempts = %s,
                error_message = %s,
                completed_at = %s
            WHERE id = %s
        """, (
            original["status"],
            original["attempts"],
            original["error_message"],
            original["completed_at"],
            contact_id,
        ))

        conn.commit()

    finally:
        cursor.close()
        conn.close()

    print("\nOriginal contact state restored automatically.")
# ------------------------------------------------------------
# TEST — Section 8-2 Contacts page detection
# ------------------------------------------------------------

# This test verifies:
#
#   _contacts_page_is_open(driver)
#
# Expected behavior:
#
#   True  -> when the Add Contact button is visible.
#   False -> when the Add Contact button is not visible.
#
# This test does NOT:
#   - add a contact
#   - modify the database
#   - click the Add Contact button
#   - open/close any Eitaa window


print("Testing current Eitaa UI state...")
print()

# ------------------------------------------------------------
# 1. Check the current state
# ------------------------------------------------------------

contacts_open = _contacts_page_is_open(driver)

print(f"_contacts_page_is_open(driver) returned: {contacts_open}")

# ------------------------------------------------------------
# 2. Inspect the Add Contact button directly
# ------------------------------------------------------------

elements = driver.find_elements(
    *Selectors.ADD_CONTACT_BUTTON
)

visible_elements = [
    element
    for element in elements
    if element.is_displayed()
]

print(f"Add Contact elements found : {len(elements)}")
print(f"Visible Add Contact buttons : {len(visible_elements)}")
print()

# ------------------------------------------------------------
# 3. Verify consistency
# ------------------------------------------------------------

expected_result = len(visible_elements) > 0

assert contacts_open == expected_result, (
    "TEST FAILED: _contacts_page_is_open() returned "
    f"{contacts_open}, but direct inspection indicates "
    f"{expected_result}."
)

# ------------------------------------------------------------
# 4. Test result
# ------------------------------------------------------------

print("Direct UI inspection agrees with the function result.")
print()

print("============================================================")
print("TEST PASSED — Section 8-2 Contacts page detection")
print("============================================================")
# ------------------------------------------------------------
# TEST — Section 8-3 Open Contacts area
# ------------------------------------------------------------

# This test verifies:
#
#   _open_contacts_area(driver)
#
# Expected behavior:
#
#   - If Contacts is already open, the function should return
#     without unnecessarily navigating again.
#   - The Contacts page should be open after the function.
#   - The Add Contact button should be visible.
#
# This test does NOT:
#   - add a contact
#   - modify the database
#   - open the Add Contact form

print("Testing _open_contacts_area(driver)...")
print()

# Record the database state of one contact so we can verify
# that this function does not modify the database.
connection = get_db_connection()

try:
    cursor = connection.cursor(dictionary=True)

    cursor.execute("""
        SELECT id, status, attempts, error_message, completed_at
        FROM contacts
        ORDER BY id
        LIMIT 1
    """)

    before_db = cursor.fetchone()

finally:
    cursor.close()
    connection.close()

# Call the Section 8 function.
_open_contacts_area(driver)

# Verify that Contacts is open.
contacts_open = _contacts_page_is_open(driver)

print(f"_open_contacts_area(driver) completed.")
print(f"Contacts page open: {contacts_open}")

# Verify that the Add Contact button is visible.
elements = driver.find_elements(*Selectors.ADD_CONTACT_BUTTON)

visible_elements = [
    element
    for element in elements
    if element.is_displayed()
]

print(f"Add Contact elements found : {len(elements)}")
print(f"Visible Add Contact buttons : {len(visible_elements)}")
print()

assert contacts_open is True, (
    "TEST FAILED: Contacts page is not open after "
    "_open_contacts_area(driver)."
)

assert len(visible_elements) > 0, (
    "TEST FAILED: Add Contact button is not visible after "
    "_open_contacts_area(driver)."
)

# Verify that the Add Contact form was NOT opened.
popup_elements = driver.find_elements(
    *Selectors.CLOSE_CONTACT_WINDOW
)

visible_popups = [
    element
    for element in popup_elements
    if element.is_displayed()
]

assert len(visible_popups) == 0, (
    "TEST FAILED: Add Contact form was unexpectedly opened."
)

# Verify that the database was not modified.
connection = get_db_connection()

try:
    cursor = connection.cursor(dictionary=True)

    cursor.execute("""
        SELECT id, status, attempts, error_message, completed_at
        FROM contacts
        WHERE id = %s
    """, (before_db["id"],))

    after_db = cursor.fetchone()

finally:
    cursor.close()
    connection.close()

assert after_db == before_db, (
    "TEST FAILED: Database state changed during "
    "_open_contacts_area(driver)."
)

print("Contacts page is open.")
print("Add Contact button is visible.")
print("Add Contact form was not opened.")
print("Database state was not modified.")
print()

print("============================================================")
print("TEST PASSED — Section 8-3 Open Contacts area")
print("============================================================")

# ------------------------------------------------------------
# TEST — Section 8-4 Detect unregistered-phone result
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TEST — Section 8-4 Detect unregistered-phone result")
print("=" * 60)

try:
    _open_contacts_area(driver)
    print("✓ Contacts area is open")

    wait_and_click(driver, Selectors.ADD_CONTACT_BUTTON)
    wait_visible(driver, Selectors.FIRST_NAME)
    print("✓ Add Contact window opened")

    test_name = "Section 8-4 Test"
    wait_and_type(driver, Selectors.FIRST_NAME, test_name)
    print(f"✓ Test name entered: {test_name}")

    test_phone = "9111111111"
    wait_and_type(driver, Selectors.PHONE, test_phone)
    print(f"✓ Test phone entered: {test_phone}")

    wait_and_click(driver, Selectors.SAVE_BUTTON)
    print("✓ SAVE button clicked")

    # Wait until the unregistered-phone result appears.
    result = wait_until(
        driver,
        lambda d: _get_unregistered_result(d),
        timeout=WAIT_SECONDS,
    )

    if not result:
        raise AssertionError(
            "Expected _get_unregistered_result() to return True "
            "for the unregistered test phone."
        )

    print("✓ Unregistered-phone toast detected")
    print(f"Detection result: {result}")
    print("✓ _get_unregistered_result() returned True")

    # Confirm the expected message is still detectable.
    if not _get_unregistered_result(driver):
        raise AssertionError(
            "Expected the unregistered-phone message to be detected."
        )

    print("✓ Expected unregistered-phone message confirmed")

    print("\n" + "-" * 60)
    print("Full report")
    print("-" * 60)
    print("Contacts area        : PASS")
    print("Add Contact window   : PASS")
    print("Name entry           : PASS")
    print("Phone entry          : PASS")
    print("SAVE                 : PASS")
    print("Toast detection      : PASS")
    print("Message verification : PASS")
    print("-" * 60)
    print("TEST PASSED")
    print("-" * 60)

except Exception as exc:
    print("\n" + "-" * 60)
    print("Full report")
    print("-" * 60)
    print("TEST FAILED")
    print(f"{type(exc).__name__}: {exc}")
    print("-" * 60)
    raise

finally:
    try:
        _close_add_contact_window(driver)

        # Verify that the Add Contact window is actually closed.
        wait_popup_closed(
            driver,
            Selectors.CLOSE_CONTACT_WINDOW,
            timeout=5
        )

        print("✓ Cleanup completed")
        print("✓ Add Contact window closed")
        print("✓ Returned to Contacts page")

    except Exception as cleanup_exc:
        print(
            f"Cleanup warning: "
            f"{type(cleanup_exc).__name__}: {cleanup_exc}"
        )
# ------------------------------------------------------------
# TEST — Section 8-5 Detect result after Save
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TEST — Section 8-5 Detect result after Save")
print("=" * 60)

try:
    _open_contacts_area(driver)
    print("✓ Contacts area is open")

    wait_and_click(driver, Selectors.ADD_CONTACT_BUTTON)
    wait_visible(driver, Selectors.FIRST_NAME)
    print("✓ Add Contact window opened")

    test_name = "Section 8-5 Test"
    wait_and_type(driver, Selectors.FIRST_NAME, test_name)
    print(f"✓ Test name entered: {test_name}")

    test_phone = "9111111111"
    wait_and_type(driver, Selectors.PHONE, test_phone)
    print(f"✓ Test phone entered: {test_phone}")

    wait_and_click(driver, Selectors.SAVE_BUTTON)
    print("✓ SAVE button clicked")

    # --------------------------------------------------------
    # Test _wait_for_add_contact_result()
    # --------------------------------------------------------
    result = _wait_for_add_contact_result(driver)

    print(f"Detected result: {result}")

    if result != "skipped":
        raise AssertionError(
            "Expected _wait_for_add_contact_result() to return "
            f"'skipped', but received {result!r}"
        )

    print("✓ Result correctly classified as: skipped")

    # --------------------------------------------------------
    # Confirm that the unregistered result was actually detected
    # --------------------------------------------------------
    if not _get_unregistered_result(driver):
        raise AssertionError(
            "Expected the unregistered-phone result to remain "
            "detectable after _wait_for_add_contact_result()."
        )

    print("✓ Unregistered result remains confirmed")

    print("\n" + "-" * 60)
    print("Full report")
    print("-" * 60)
    print("Contacts area        : PASS")
    print("Add Contact window   : PASS")
    print("Name entry           : PASS")
    print("Phone entry          : PASS")
    print("SAVE                 : PASS")
    print("Result detection     : PASS")
    print("Result classification: PASS")
    print("Expected result      : skipped")
    print("-" * 60)
    print("TEST PASSED")
    print("-" * 60)

except Exception as exc:
    print("\n" + "-" * 60)
    print("Full report")
    print("-" * 60)
    print("TEST FAILED")
    print(f"{type(exc).__name__}: {exc}")
    print("-" * 60)
    raise

finally:
    # --------------------------------------------------------
    # Cleanup
    #
    # The unregistered result can leave the Add Contact window
    # open. Close it and verify that it is actually closed.
    # --------------------------------------------------------
    try:
        _close_add_contact_window(driver)

        wait_popup_closed(
            driver,
            Selectors.CLOSE_CONTACT_WINDOW,
            timeout=5
        )

        print("✓ Cleanup completed")
        print("✓ Add Contact window closed")
        print("✓ Returned to Contacts page")

    except Exception as cleanup_exc:
        print(
            f"Cleanup warning: "
            f"{type(cleanup_exc).__name__}: {cleanup_exc}"
        )
# ------------------------------------------------------------
# TEST — Section 8-6 Close Add Contact window
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TEST — Section 8-6 Close Add Contact window")
print("=" * 60)

try:
    _open_contacts_area(driver)
    print("✓ Contacts area is open")

    wait_and_click(driver, Selectors.ADD_CONTACT_BUTTON)
    wait_visible(driver, Selectors.FIRST_NAME)
    print("✓ Add Contact window opened")

    test_name = "Section 8-6 Test"
    wait_and_type(driver, Selectors.FIRST_NAME, test_name)
    print(f"✓ Test name entered: {test_name}")

    test_phone = "9111111111"
    wait_and_type(driver, Selectors.PHONE, test_phone)
    print(f"✓ Test phone entered: {test_phone}")

    wait_and_click(driver, Selectors.SAVE_BUTTON)
    print("✓ SAVE button clicked")

    # --------------------------------------------------------
    # Wait for the unregistered-phone result.
    # This creates the same UI state in which the close helper
    # may need to perform the second close action.
    # --------------------------------------------------------
    result = wait_until(
        driver,
        lambda d: _get_unregistered_result(d),
        timeout=WAIT_SECONDS,
    )

    if not result:
        raise AssertionError(
            "Expected the unregistered-phone result before "
            "testing _close_add_contact_window()."
        )

    print("✓ Unregistered-phone result detected")

    # Confirm that the Add Contact close button is visible
    # before calling the helper.
    wait_visible(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
        timeout=5
    )

    print("✓ Add Contact window is still open")

    # --------------------------------------------------------
    # Test the actual Section 8-6 helper.
    # --------------------------------------------------------
    _close_add_contact_window(driver)

    print("✓ _close_add_contact_window() executed")

    # --------------------------------------------------------
    # Verify that the Add Contact window is actually closed.
    # --------------------------------------------------------
    wait_popup_closed(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
        timeout=5
    )

    print("✓ Add Contact window closed")

    print("\n" + "-" * 60)
    print("Full report")
    print("-" * 60)
    print("Contacts area        : PASS")
    print("Add Contact window   : PASS")
    print("Name entry           : PASS")
    print("Phone entry          : PASS")
    print("SAVE                 : PASS")
    print("Unregistered result  : PASS")
    print("Close helper         : PASS")
    print("Window closed        : PASS")
    print("-" * 60)
    print("TEST PASSED")
    print("-" * 60)

except Exception as exc:
    print("\n" + "-" * 60)
    print("Full report")
    print("-" * 60)
    print("TEST FAILED")
    print(f"{type(exc).__name__}: {exc}")
    print("-" * 60)
    raise

finally:
    # --------------------------------------------------------
    # Final safety cleanup.
    # If the test failed before the window was closed,
    # make one final attempt to leave the browser clean.
    # --------------------------------------------------------
    try:
        _close_add_contact_window_if_open(driver)

        wait_popup_closed(
            driver,
            Selectors.CLOSE_CONTACT_WINDOW,
            timeout=5
        )

        print("✓ Final cleanup completed")
        print("✓ Returned to Contacts page")

    except Exception as cleanup_exc:
        print(
            f"Cleanup warning: "
            f"{type(cleanup_exc).__name__}: {cleanup_exc}"
        )
# ------------------------------------------------------------
# TEST — Section 8-7 Best-effort cleanup after operational failure
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TEST — Section 8-7 Best-effort cleanup after operational failure")
print("=" * 60)

try:
    # --------------------------------------------------------
    # 1. Open Contacts area
    # --------------------------------------------------------
    _open_contacts_area(driver)
    print("✓ Contacts area is open")

    # --------------------------------------------------------
    # 2. Open Add Contact window
    # --------------------------------------------------------
    wait_and_click(driver, Selectors.ADD_CONTACT_BUTTON)

    wait_visible(
        driver,
        Selectors.FIRST_NAME,
    )

    print("✓ Add Contact window opened")

    # --------------------------------------------------------
    # 3. Confirm the close button is visible
    # --------------------------------------------------------
    wait_visible(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
    )

    print("✓ Add Contact close button is visible")

    # --------------------------------------------------------
    # 4. Test _close_add_contact_window_if_open()
    # --------------------------------------------------------
    _close_add_contact_window_if_open(driver)

    print("✓ _close_add_contact_window_if_open() executed")

    # --------------------------------------------------------
    # 5. Verify that the window was actually closed
    # --------------------------------------------------------
    wait_popup_closed(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
        timeout=5,
    )

    print("✓ Add Contact window closed")

    # --------------------------------------------------------
    # 6. Call the helper again while the window is already
    #    closed. It should safely do nothing.
    # --------------------------------------------------------
    _close_add_contact_window_if_open(driver)

    print("✓ Cleanup helper safely handled already-closed window")

    # --------------------------------------------------------
    # 7. Final verification
    # --------------------------------------------------------
    wait_popup_closed(
        driver,
        Selectors.CLOSE_CONTACT_WINDOW,
        timeout=5,
    )

    print("✓ Add Contact window remains closed")

    print("\n" + "-" * 60)
    print("Full report")
    print("-" * 60)
    print("Contacts area        : PASS")
    print("Add Contact window   : PASS")
    print("Close button visible : PASS")
    print("Cleanup helper       : PASS")
    print("Window closed        : PASS")
    print("Already-closed case  : PASS")
    print("-" * 60)
    print("TEST PASSED")
    print("-" * 60)

except Exception as exc:
    print("\n" + "-" * 60)
    print("Full report")
    print("-" * 60)
    print("TEST FAILED")
    print(f"{type(exc).__name__}: {exc}")
    print("-" * 60)
    raise

finally:
    # --------------------------------------------------------
    # Final safety cleanup.
    # The helper itself is intentionally best-effort.
    # --------------------------------------------------------
    try:
        _close_add_contact_window_if_open(driver)

        wait_popup_closed(
            driver,
            Selectors.CLOSE_CONTACT_WINDOW,
            timeout=5,
        )

        print("✓ Final cleanup completed")
        print("✓ Returned to Contacts page")

    except Exception as cleanup_exc:
        print(
            f"Cleanup warning: "
            f"{type(cleanup_exc).__name__}: {cleanup_exc}"
        )
# ------------------------------------------------------------
# TEST — Section 8-8 Main add_contact() function
# ------------------------------------------------------------

TEST_PHONE_NORMALIZED = "9112345678"
EXPECTED_PHONE_DIGITS = "989112345678"

try:
    wait_and_click(driver, Selectors.ADD_CONTACT_BUTTON)

    wait_and_type(
        driver,
        Selectors.FIRST_NAME,
        "Section 8-8 Test",
    )

    _enter_phone_number(driver, TEST_PHONE_NORMALIZED)

    phone_field = wait_visible(driver, Selectors.PHONE)

    # Eitaa uses a contenteditable DIV, not an input element.
    actual_value = phone_field.text.strip()

    actual_digits = "".join(
        character
        for character in actual_value
        if character.isdigit()
    )

    print(f"Input number:    {TEST_PHONE_NORMALIZED}")
    print(f"Expected digits: {EXPECTED_PHONE_DIGITS}")
    print(f"Displayed text:  {actual_value}")
    print(f"Actual digits:   {actual_digits}")

    if actual_digits == EXPECTED_PHONE_DIGITS:
        print("\nPASS: Country prefix and phone number are correct.")
    else:
        print("\nFAIL: The resulting phone number does not match.")

except Exception as exc:
    print("\nFAIL: Section 8-8 test raised an exception.")
    print(f"{type(exc).__name__}: {exc}")

finally:
    try:
        _close_add_contact_window_if_open(driver)
        print("\nTest dialog cleanup completed.")
    except Exception as cleanup_exc:
        print(
            f"\nCleanup warning: "
            f"{type(cleanup_exc).__name__}: {cleanup_exc}"
        )

print("=" * 60)
print("Section 8-8 test finished.")
print("=" * 60)

In [9]:
# ============================================================
# 9- Fetching the Next Contact
# ============================================================

def fetch_next_contact():
    """
    Fetch the next eligible contact from the contacts table.

    Eligible contacts are:
        - pending contacts
        - failed contacts with fewer than 3 attempts

    This function ONLY reads the database.

    It does NOT:
        - change the contact status
        - increment attempts
        - retry contacts
        - call add_contact()
        - interact with Selenium

    Returns
    -------
    tuple | None
        A tuple containing:

            (
                contact_id,
                full_name,
                phone_normalized,
            )

        Returns None when no eligible contact remains.

    Notes
    -----
    Section 10 is responsible for:
        - calling add_contact()
        - managing attempts
        - handling retries
        - deciding when processing is complete
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT
                id,
                full_name,
                phone_normalized
            FROM contacts
            WHERE
                status = 'pending'
                OR (
                    status = 'failed'
                    AND attempts < 3
                )
            ORDER BY id ASC
            LIMIT 1
            """
        )

        contact = cursor.fetchone()

        if contact is None:
            return None

        return (
            contact[0],  # contact_id
            contact[1],  # full_name
            contact[2],  # phone_normalized
        )

    finally:
        cursor.close()
        conn.close()

In [ ]:
DELETE
ME!
# ============================================================
# Test - Section 9: Fetching the Next Contact
# ============================================================

print()
print("============================================================")
print("TEST - SECTION 9: FETCHING THE NEXT CONTACT")
print("============================================================")

# ------------------------------------------------------------
# 1. Fetch the next contact using Section 9
# ------------------------------------------------------------

contact = fetch_next_contact()

# ------------------------------------------------------------
# 2. Verify the returned value
# ------------------------------------------------------------

if contact is None:
    print("✓ fetch_next_contact() returned None.")
    print("  No eligible contacts are currently available.")

else:
    print("✓ fetch_next_contact() returned a contact.")

    # --------------------------------------------------------
    # Verify that exactly three values were returned
    # --------------------------------------------------------

    if not isinstance(contact, tuple):
        raise AssertionError(
            f"Expected a tuple, got {type(contact).__name__}"
        )

    if len(contact) != 3:
        raise AssertionError(
            f"Expected 3 values, got {len(contact)}"
        )

    print("✓ Return value contains exactly 3 fields.")

    # --------------------------------------------------------
    # Extract the contact data
    # --------------------------------------------------------

    contact_id, full_name, phone_normalized = contact

    # --------------------------------------------------------
    # Verify the returned values
    # --------------------------------------------------------

    if not isinstance(contact_id, int):
        raise AssertionError(
            f"contact_id should be int, got {type(contact_id).__name__}"
        )

    if not isinstance(full_name, str) or not full_name.strip():
        raise AssertionError(
            "full_name should be a non-empty string."
        )

    if not isinstance(phone_normalized, str):
        raise AssertionError(
            "phone_normalized should be a string."
        )

    print("✓ contact_id is valid.")
    print("✓ full_name is valid.")
    print("✓ phone_normalized is valid.")

    # --------------------------------------------------------
    # Verify the returned contact against the database
    # --------------------------------------------------------

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT
                id,
                full_name,
                phone_normalized,
                status,
                attempts
            FROM contacts
            WHERE id = %s
            """,
            (contact_id,),
        )

        db_contact = cursor.fetchone()

    finally:
        cursor.close()
        conn.close()

    if db_contact is None:
        raise AssertionError(
            f"Contact id={contact_id} was not found in the database."
        )

    db_id = db_contact[0]
    db_full_name = db_contact[1]
    db_phone = db_contact[2]
    db_status = db_contact[3]
    db_attempts = db_contact[4]

    # --------------------------------------------------------
    # Verify returned data matches the database
    # --------------------------------------------------------

    if contact_id != db_id:
        raise AssertionError(
            "Returned contact_id does not match the database."
        )

    if full_name != db_full_name:
        raise AssertionError(
            "Returned full_name does not match the database."
        )

    if phone_normalized != db_phone:
        raise AssertionError(
            "Returned phone_normalized does not match the database."
        )

    print("✓ Returned contact data matches the database.")

    # --------------------------------------------------------
    # Verify eligibility
    # --------------------------------------------------------

    is_eligible = (
            db_status == "pending"
            or (
                    db_status == "failed"
                    and db_attempts < 3
            )
    )

    if not is_eligible:
        raise AssertionError(
            f"Returned contact is not eligible: "
            f"status={db_status!r}, attempts={db_attempts}"
        )

    print("✓ Returned contact is eligible.")
    print(f"  status   : {db_status}")
    print(f"  attempts : {db_attempts}")

    # --------------------------------------------------------
    # Display the contact returned by Section 9
    # --------------------------------------------------------

    print()
    print("---------- FETCHED CONTACT ----------")
    print(f"contact_id       : {contact_id}")
    print(f"full_name        : {full_name}")
    print(f"phone_normalized : {phone_normalized}")
    print(f"status           : {db_status}")
    print(f"attempts         : {db_attempts}")
    print("-------------------------------------")

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print()
print("============================================================")
print("✓ SECTION 9 TEST PASSED")
print("============================================================")


In [10]:
# ============================================================
# 10- Main Processing Loop
# ============================================================

MAX_ATTEMPTS = 3


# ------------------------------------------------------------
# 10-1 Increment contact attempt counter
# ------------------------------------------------------------

def _increment_contact_attempt(contact_id):
    """
    Increment the attempts counter for one contact.

    Attempt management belongs exclusively to Section 10.
    Section 8 performs the actual Eitaa attempt but does not
    modify the attempts counter.
    """

    conn = get_db_connection()

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            UPDATE contacts
            SET
                attempts = attempts + 1,
                started_at = NOW()
            WHERE id = %s
            """,
            (contact_id,),
        )

        conn.commit()

    except Exception:
        conn.rollback()
        raise

    finally:
        cursor.close()
        conn.close()


# ------------------------------------------------------------
# 10-2 Main processing loop
# ------------------------------------------------------------

def process_all_contacts():
    """
    Process all eligible contacts.

    Workflow:
        1. Fetch the next eligible contact using Section 9.
        2. Increment its attempt counter.
        3. Process exactly one attempt using Section 8.
        4. Continue to the next contact.
        5. Failed contacts are retried until they reach
           MAX_ATTEMPTS.

    Section 8 is responsible for recording:
        - success
        - skipped
        - failed

    Section 10 is responsible for:
        - attempt counting
        - retry decisions
        - main loop control
    """

    total_processed = 0
    success_count = 0
    skipped_count = 0
    failed_count = 0

    print()
    print("========== SECTION 10 ==========")
    print("Starting main contact-processing loop...")
    print()

    while True:

        # ----------------------------------------------------
        # 1. Fetch the next eligible contact
        # ----------------------------------------------------

        contact = fetch_next_contact()

        if contact is None:
            print("No eligible contacts remain.")
            break

        contact_id, full_name, phone_normalized = contact

        # ----------------------------------------------------
        # 2. Increment attempt counter
        # ----------------------------------------------------

        _increment_contact_attempt(contact_id)

        # ----------------------------------------------------
        # 3. Determine current attempt number
        #
        # fetch_next_contact() does not return attempts, so
        # Section 10 reads it only after claiming the contact.
        # ----------------------------------------------------

        conn = get_db_connection()

        try:
            cursor = conn.cursor()

            cursor.execute(
                """
                SELECT attempts
                FROM contacts
                WHERE id = %s
                """,
                (contact_id,),
            )

            row = cursor.fetchone()

            if row is None:
                raise RuntimeError(
                    f"Contact disappeared from database: "
                    f"contact_id={contact_id}"
                )

            attempt_number = row[0]

        finally:
            cursor.close()
            conn.close()

        # ----------------------------------------------------
        # 4. Process exactly ONE attempt
        # ----------------------------------------------------

        print(
            f"Processing contact_id={contact_id} "
            f"(attempt {attempt_number}/{MAX_ATTEMPTS})"
        )

        result = add_contact(
            contact_id=contact_id,
            full_name=full_name,
            phone_normalized=phone_normalized,
        )

        total_processed += 1

        # ----------------------------------------------------
        # 5. Handle Section 8 result
        # ----------------------------------------------------

        if result == "success":

            success_count += 1

            print(
                f"SUCCESS: contact_id={contact_id}"
            )

        elif result == "skipped":

            skipped_count += 1

            print(
                f"SKIPPED: contact_id={contact_id}"
            )

        elif result == "failed":

            failed_count += 1

            if attempt_number < MAX_ATTEMPTS:
                print(
                    f"FAILED: contact_id={contact_id} "
                    f"-> will be retried "
                    f"(attempt {attempt_number}/{MAX_ATTEMPTS})"
                )
            else:
                print(
                    f"FAILED: contact_id={contact_id} "
                    f"-> maximum attempts reached "
                    f"({MAX_ATTEMPTS})"
                )

        else:
            # Defensive guard. Section 8 should only return
            # success, skipped, or failed.
            raise RuntimeError(
                f"Unexpected result from Section 8: {result!r}"
            )

        print()

    # --------------------------------------------------------
    # 6. Final processing summary
    # --------------------------------------------------------

    print("========== SECTION 10 RESULT ==========")
    print(f"Total attempts : {total_processed}")
    print(f"Success        : {success_count}")
    print(f"Skipped        : {skipped_count}")
    print(f"Failed         : {failed_count}")
    print("========================================")


# ------------------------------------------------------------
# 10-3 Start processing
# ------------------------------------------------------------

process_all_contacts()


========== SECTION 10 ==========
Starting main contact-processing loop...

No eligible contacts remain.
========== SECTION 10 RESULT ==========
Total attempts : 0
Success        : 0
Skipped        : 0
Failed         : 0
